# Data-informed section slopes: upper-reference prototype

**2 October 2026 — all 16 cases.** Review an explicitly steeper reference choice, its observational limits and a graph-derived alternative to arbitrary compound slope-floor fractions. This notebook reads results from `RAPID/investigations/data_informed_slope_method_audit.py`; it does not promote any profile or change hydraulic parameters.

**The fixed-window curves below are stress tests, not recommended replacements.** In particular, case 11 and case 15 show material dependence on section scale. Sections must be reviewed around profile breaks before using a bulk slope as an endpoint head-drop target.

Full [method, assumptions and findings](../assets/data_informed_slope_method_20261002/README.md). No new RAPID simulations or resistance computations were performed.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src').is_dir())
OUT=ROOT/'RAPID/assets/data_informed_slope_method_20261002'
sources=json.loads((OUT/'provenance.json').read_text())['sources']
def read(name):
    d=pd.read_csv(OUT/name,dtype={'example':str});d['example']=d.example.str.zfill(2);return d
sections=read('section_evidence.csv');profiles=read('candidate_sections.csv');cases=read('candidate_cases.csv')
domains=read('compound_maxmin.csv');heldout=read('heldout_cycle_scores.csv');shifted=read('shifted_windows.csv')
pio.renderers.default='plotly_mimetype'
assert cases.example.nunique()==16 and len(domains)==179*8
assert domains.certificate_min_slack_m.min()>=-1e-8
display(cases[cases.target=='p75'][['example','scale','evidence_length_pct','head_drop_ratio','min_evidence_slope','n_evidence_below_1e5']])

## What is the proposed method?

1. Define approximately consistent-gradient sections, separating major breaks. Estimate cycle-specific slopes/head drops with equal cycle weight and sufficient spatial coverage.
2. Make the reference condition explicit: P50 for typical, P75 as an upper-reference candidate; P90 as a more extreme sensitivity. A percentile is a choice of condition, not a physical minimum.
3. Add section head-drop targets to the existing longitudinal objective. Use a separate, conservatively supported lower bound only where data resolve it; weaken targets where cycles disagree. Do not assume robust regression slope × length equals endpoint drop across a knickpoint.
4. Transfer local targets to compound edges and preserve the selected boundary heads. Determine the graph-feasible target fraction instead of picking an arbitrary widening factor. Then assess a head allocation closest to the target under that envelope.
5. Flag unsupported sections and conflicts. There is no guarantee that data-supported slopes are always above $10^{-5}$.

The prototype curves simply integrate section P50/P75 targets where the lower 5th bootstrap percentile of the empirical cycle P10 is positive (2,000 cycle resamples, >=5 cycles). Elsewhere they preserve the old section endpoint drop. This stricter exploratory evidence check does not establish final section validity. The candidate outlet head is fixed and the inlet head is free, so the table exposes total head-loss changes.

### What was tested?

Leave-one-cycle-out slope quantile predictions, early/late cycle differences, bootstrap and jackknife sensitivity, shifted section windows, 10/20 km candidate profiles, and 1,432 compound graph-feasibility calculations. These are **not** held-out validation of a new WSE fit. Coverage rules, fallback use and graph assumptions are documented in the linked report.

In [ ]:
display(heldout[heldout.kind=='10km'].groupby('quantile')[['mae_scaled','heldout_below_target']].median())
display(sections[(sections.example=='11')&(sections.kind=='reviewed')][['section','n_cycles','p10','p50','p75','p90','p75_boot_lo','p75_boot_hi','early_p75','late_p75','epoch_ratio']])
display(shifted[shifted.n_cycles>=5].groupby('example').ratio_to_unshifted.agg(['min','median','max']))

## Per-case candidate profiles

Red: current promoted profile. Blue: P50 candidate. Orange: P75 candidate. Dashed orange: P75 with 20 km rather than 10 km sections. Gray shading marks 10 km windows without a positive lower-tail evidence diagnostic; these preserve current section drops. Because integration starts at the outlet, candidate elevations upstream of a changed section also shift.

The displays deliberately show where using regression slopes as piecewise-linear gradients changes head loss. They should guide section review, not be promoted directly.

In [ ]:
def plot_case(ex):
    old=pd.read_parquet(Path(sources[ex])/'reference_profile.parquet')
    fig=go.Figure()
    fig.add_scatter(x=old.reference_section_dist_out_m/1000,y=old.fitted_wse_m,name='Current promoted profile',line=dict(color='#d62728',width=3))
    for scale,target,color,dash in [('10km','p50','#2563eb','solid'),('10km','p75','#f97316','solid'),('20km','p75','#f97316','dash')]:
        g=profiles[(profiles.example==ex)&(profiles.scale==scale)&(profiles.target==target)].sort_values('lo')
        x=np.r_[g.lo.to_numpy(),g.hi.iloc[-1]]/1000;y=np.r_[g.head_lo.to_numpy(),g.head_hi.iloc[-1]]
        fig.add_scatter(x=x,y=y,mode='lines+markers',name=f'{target.upper()} candidate, {scale}',line=dict(color=color,dash=dash),marker=dict(size=3))
        if scale=='10km' and target=='p75':
            for r in g[~g.evidence_based].itertuples():fig.add_vrect(x0=r.lo/1000,x1=r.hi/1000,fillcolor='gray',opacity=.15,line_width=0,layer='below')
    fig.update_layout(title=f'Case {ex}: candidate profile stress tests — not adopted',xaxis_title='Distance to outlet (km)',yaxis_title='WSE (m)',height=470,template='plotly_white')
    fig.show()
    display(cases[(cases.example==ex)&(cases.target=='p75')][['scale','evidence_length_pct','head_drop_ratio','min_evidence_slope']])
    display(sections[(sections.example==ex)&(sections.kind=='10km')][['lo','hi','n_cycles','p50','p75','lower_evidence_bound','epoch_ratio','jackknife_relative_range']])

### Case 02

In [ ]:
plot_case("02")

### Case 03

In [ ]:
plot_case("03")

### Case 05

In [ ]:
plot_case("05")

### Case 11

In [ ]:
plot_case("11")

### Case 13

In [ ]:
plot_case("13")

### Case 15

In [ ]:
plot_case("15")

### Case 21

In [ ]:
plot_case("21")

### Case 22

In [ ]:
plot_case("22")

### Case 37

In [ ]:
plot_case("37")

### Case 41

In [ ]:
plot_case("41")

### Case 45

In [ ]:
plot_case("45")

### Case 46

In [ ]:
plot_case("46")

### Case 48

In [ ]:
plot_case("48")

### Case 54

In [ ]:
plot_case("54")

### Case 55

In [ ]:
plot_case("55")

### Case 56

In [ ]:
plot_case("56")

## Compound units: maximum feasible common target fraction

For local positive target $T_e$, physical length $L_e$, and fixed boundary drop $\Delta H$:

$$\rho_{\max}=\frac{\Delta H}{\max_p\sum_{e\in p}T_e L_e}.$$

Every source–sink path must accommodate the same boundary drop. This longest-weighted-path calculation yields the largest feasible common lower envelope $S_e\geq\rho_{\max}T_e$. The script checks a constructive head certificate for every domain. It is a **graph regularization choice**, not an observed minimum slope; maximizing the minimum may constrain some paths tightly. Selecting an optimal internal allocation and testing its hydraulic consequences remain outstanding.

The proposed second-stage solve would minimize the current slope-deviation objective within the feasible envelope, rather than use the extreme certificate as its final solution. Partly unsupported domains inherit old profile targets explicitly. No dimensional slope floor is invented here, but those fallbacks and the target condition remain assumptions.

In [ ]:
selected=domains[(domains.scale=='10km')&(domains.target=='p75')]
display(selected.groupby('boundary').max_feasible_target_fraction.describe())
display(selected.groupby(['example','boundary']).agg(domains=('domain','size'),fully_evidenced=('all_links_evidenced','sum'),min_fraction=('max_feasible_target_fraction','min'),median_fraction=('max_feasible_target_fraction','median')))
display(domains[(domains.example=='15')&(domains.domain==4)])

## Decision after this audit

The data-informed direction is promising, but the automatic fixed-window implementation is **not ready to replace production**. Resolve section boundaries and temporal reference choice, fit the actual longitudinal objective with withheld-cycle checks, and only then finalize compound regularization and return to Kb/stability. Case 11 still has plausible very-low gradients; case 22 has broad unresolved coverage. A rule guaranteeing their elimination would add an assumption beyond these observations.